# Analysis Examples for Dense Polymer Melts

This notebook analyzes a dense polymer melt trajectory with [MDAnalysis](https://www.mdanalysis.org/). By default it uses the NPT trajectory produced by `2_Simulating_Dense_Polymer_Melts.ipynb`, converted from MuPT with `primitive_to_mdanalysis`.

It shows common analyses such as:

1. Radius of gyration of the chains
2. Mean-squared displacement (MSD)
3. Radial distribution function (RDF), total and intermolecular
4.
5.
6.

**Settings at the top of the next cell:**
- `TRAJECTORY_SOURCE`: `'openmm'` (notebook 2 output, default) or `'lammps'` (a LAMMPS data file + `.lammpstrj` dump, e.g. from a run started with the LAMMPS files exported in notebook 2).
- `TRAJECTORY_IS_LINEAR`: `True` if frames are written at equal time intervals, `False` for non-linearly (e.g. log-) spaced dumps.

Each section is self-contained, so new analyses can be added at the end.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem

import MDAnalysis as mda
from MDAnalysis import transformations as trans
from MDAnalysis.analysis.msd import EinsteinMSD
from MDAnalysis.analysis.rdf import InterRDF

# Make the repo's `utilities` package importable (same as notebook #2_Simulating_Dense_Polymer_Melts.ipynb)
EXAMPLES_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'examples_system').is_dir() and (candidate / 'recipes').is_dir()
)
if str(EXAMPLES_ROOT) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_ROOT))

from mupt.temporary.sdf import primitive_from_mupt_sdf
from mupt.interfaces.mdanalysis import primitive_to_mdanalysis
from utilities.notebook import load_manifest

# warnings.filterwarnings('ignore', message='DCDReader currently makes independent timesteps')

# ---- Settings -----------------------------------------------------------------
TRAJECTORY_SOURCE = 'openmm'    # 'openmm' (notebook 2 output) or 'lammps'
TRAJECTORY_IS_LINEAR = True     # False for non-linearly spaced dumps
# -------------------------------------------------------------------------------

# Files saved by notebooks 1 and 2
OUTPUT_DIR = EXAMPLES_ROOT / 'examples_system' / 'dense_melt_demo_outputs'
MANIFEST_PATH = OUTPUT_DIR / 'dense_melt_manifest.json'
manifest = load_manifest(MANIFEST_PATH)
recipe_name = manifest['recipe_name']

if TRAJECTORY_SOURCE == 'openmm':
    # Rebuild the MuPT system from disk (notebook 3 starts with a fresh kernel)
    TRAJECTORY = OUTPUT_DIR / f'{recipe_name}_openmm_npt.dcd'
    resname_map = manifest['resname_map']
    mupt_universe = primitive_from_mupt_sdf(Path(manifest['sdf_path']))
    print(f'Loaded {manifest["sdf_path"]}')

elif TRAJECTORY_SOURCE == 'lammps':
    # Topology: LAMMPS data file exported by notebook 2 (contains masses and bonds)
    LAMMPS_DATA = OUTPUT_DIR / 'engine_starting_files' / 'lammps' / f'{recipe_name}.lmp'
    # TODO: path to your LAMMPS dump (must contain atom ids and x y z)
    TRAJECTORY = OUTPUT_DIR / 'engine_starting_files' / 'lammps' / f'{recipe_name}.lammpstrj'
    # TODO: time between dumped frames in ps (dump interval x LAMMPS timestep)
    LAMMPS_DT_PS = 1.0

else:
    raise ValueError(f'Unknown TRAJECTORY_SOURCE: {TRAJECTORY_SOURCE!r}')

print(f'Source: {TRAJECTORY_SOURCE}, trajectory: {TRAJECTORY}')

_periodic_table = Chem.GetPeriodicTable()


def load_md_universe(unwrap=False):
    """Return an MDAnalysis Universe for the chosen trajectory.
    Use unwrap=True for dynamics (e.g. MSD)."""
    if TRAJECTORY_SOURCE == 'openmm':
        u = primitive_to_mdanalysis(mupt_universe, resname_map=resname_map)
        u.load_new(str(TRAJECTORY))
        masses = [_periodic_table.GetAtomicWeight(el) for el in u.atoms.elements]
        u.add_TopologyAttr('masses', masses)
    else:  # 'lammps'
        u = mda.Universe(
            str(LAMMPS_DATA), str(TRAJECTORY),
            topology_format='DATA', format='LAMMPSDUMP', dt=LAMMPS_DT_PS,
        )

    if len(getattr(u, 'bonds', [])) == 0:
        u.atoms.guess_bonds()

    if unwrap:
        u.trajectory.add_transformations(trans.NoJump())
    return u

## 1. Radius of gyration

The radius of gyration $R_g$ measures the size of each polymer chain. MDAnalysis identifies the chains as bonded *fragments*; we compute $R_g$ for every chain in every frame and plot the average over chains.

In [ ]:
u = load_md_universe()
chains = u.atoms.fragments
print(f'{len(chains)} chains, {len(u.trajectory)} frames')

times_ps, rg_mean_nm = [], []
for ts in u.trajectory:
    rg = [chain.radius_of_gyration() for chain in chains]   # Angstrom
    times_ps.append(ts.time)
    rg_mean_nm.append(np.mean(rg) / 10.0)

plt.plot(times_ps, rg_mean_nm)
plt.xlabel('Time (ps)')
plt.ylabel('Mean chain $R_g$ (nm)')
plt.title('Radius of gyration')
plt.show()

In [ ]:
# print(np.round(rg_mean_nm[:5], 6))

## 2. Mean-squared displacement

The MSD measures how far atoms move over a lag time $\tau$:
$\mathrm{MSD}(\tau) = \langle |\mathbf{r}(t+\tau) - \mathbf{r}(t)|^2 \rangle$.

Because the trajectory is wrapped back into the periodic box, we load it with `unwrap=True` (the `NoJump` transformation) so atoms crossing the boundary don't appear to jump across the box.

If frames were dumped at non-uniform intervals (`TRAJECTORY_IS_LINEAR = False`), `EinsteinMSD` must be told so with `non_linear=True`, and it then cannot use the FFT algorithm.

In [ ]:
u = load_md_universe(unwrap=True)

if TRAJECTORY_IS_LINEAR:
    msd = EinsteinMSD(u, select='all', msd_type='xyz', fft=False)
    msd.run()
    lag_ps = np.arange(msd.n_frames) * u.trajectory.dt
else:
    # Placeholder for non-linearly dumped trajectories.
    # Requires an MDAnalysis version whose EinsteinMSD supports `non_linear`.
    msd = EinsteinMSD(u, select='all', msd_type='xyz', fft=False, non_linear=True)
    msd.run()
    lag_ps = msd.results.delta_t_values   # TODO(check): confirm units are ps

msd_A2 = msd.results.timeseries
mask = lag_ps > 0   # lag 0 can't be shown on a log axis

plt.loglog(lag_ps[mask], msd_A2[mask], marker='o', markersize=3)
plt.xlabel('Lag time (ps)')
plt.ylabel(r'MSD ($\AA^2$)')
plt.title('Mean-squared displacement (all atoms)')
plt.show()

In [ ]:
# MSD of a single atom: msds_by_particle has shape (n_lags, n_atoms)
ATOM_INDEX = 10
msd_one = msd.results.msds_by_particle[:, ATOM_INDEX]

plt.loglog(lag_ps[mask], msd_one[mask], marker='o', markersize=3)
plt.xlabel('Lag time (ps)')
plt.ylabel(r'MSD ($\AA^2$)')
plt.title(f'Mean-squared displacement (atom {ATOM_INDEX})')
plt.show()

## 3. Radial distribution function

The RDF $g(r)$ describes how the density of atoms varies with distance from a reference atom. We compute the carbon–carbon RDF, averaged over all frames, in two ways using `exclusion_block`:

- **Total**, `exclusion_block=(1, 1)`: only each atom paired with itself is removed. Without this, every atom counts itself at $r = 0$ and produces a spurious peak there.
- **Intermolecular**, `exclusion_block=(n, n)` with `n` carbons per chain: all pairs within the same chain are removed, leaving only contacts between *different* chains.

The difference between the two curves at short $r$ is the contribution of bonded and near-bonded neighbours on the same chain. Carbons are selected by mass so the same code works for both trajectory sources.

In [ ]:
u = load_md_universe()
carbons = u.select_atoms('prop mass > 11.9 and prop mass < 12.1')   # carbon by mass

# exclusion_block assumes each chain's atoms are listed contiguously
assert np.all(np.diff(carbons.fragindices) >= 0), 'atoms are not ordered by chain'
carbons_per_chain = {len(chain & carbons) for chain in u.atoms.fragments}
assert len(carbons_per_chain) == 1, 'intermolecular exclusion_block needs equal-length chains'
n = carbons_per_chain.pop()

r_max = 0.5 * u.dimensions[:3].min()   # stay within half the box
rdf_total = InterRDF(carbons, carbons, nbins=150, range=(0.0, r_max), exclusion_block=(1, 1))
rdf_inter = InterRDF(carbons, carbons, nbins=150, range=(0.0, r_max), exclusion_block=(n, n))
rdf_total.run()
rdf_inter.run()

plt.plot(rdf_total.results.bins, rdf_total.results.rdf, label='Total (self excluded)')
plt.plot(rdf_inter.results.bins, rdf_inter.results.rdf, label='Intermolecular')
plt.xlabel(r'$r$ ($\AA$)')
plt.ylabel('$g(r)$')
plt.title('C–C radial distribution function')
plt.legend()
plt.show()

## Adding your own analysis

To add an analysis, append a new section that:

1. starts with a short markdown cell explaining what is computed,
2. loads the system with `load_md_universe()` (use `unwrap=True` for dynamical quantities),
3. works for both `TRAJECTORY_SOURCE` options (e.g. select atoms by mass, not by element name),
4. ends with a plot.
5. Do not commit the outputs.